In [ ]:
# Для использования YOLO устанавливаю ultralytics
!pip install ultralytics tensorboard

In [ ]:
import numpy as np
import pandas as pd
import os, gc
from pathlib import Path


import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2

# for YOLO working
import yaml
from ultralytics import YOLO
from ultralytics.data.dataset import YOLODataset
from ultralytics.data import build_dataloader
from ultralytics.nn.tasks import DetectionModel


import matplotlib.pyplot as plt
import matplotlib.patches as patches

from sklearn.model_selection import train_test_split

from tqdm.notebook import tqdm
from IPython.display import clear_output

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
# Определяем среду выполнения
if 'google.colab' in str(get_ipython()):
    CURRENT_ENV = 'colab'
    print("Работаем в Google Colab")
    DATASET_PATH = Path("/content/datasets/wider-face-yolo")
elif 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    CURRENT_ENV = 'kaggle'
    print("Работаем в Kaggle")
    DATASET_PATH = Path("/kaggle/working/wider-face-yolo")
else:
    CURRENT_ENV = 'local'
    DATASET_PATH = Path("./wider-face-yolo")

# Детекция лиц (Face Detection)

Буду использовать датасет Wider Face с сайта Kaggle. В нём содержится 12850 фотографий.
https://www.kaggle.com/datasets/lylmsc/wider-face-for-yolo-training

In [ ]:
import kagglehub
# Скачиваем датасет через kagglehub (работает и в Colab, и в Kaggle)
path_to_dataset = kagglehub.dataset_download("lylmsc/wider-face-for-yolo-training")
print(f"Датасет скачан в: {path_to_dataset}")

In [ ]:
images_folder_path = path_to_dataset + '/images'
labels_folder_path = path_to_dataset + '/labels'

Посмотрим, что нам дано в этом наборе данных

In [ ]:
image_path = images_folder_path + '/wider_10001.jpg'
img = Image.open(image_path).convert('RGB')
plt.imshow(img)

labels_path = labels_folder_path + '/wider_10001.txt'
labels = pd.read_csv(labels_path, names=['z', 'x1', 'y1', 'x2', 'y2'], sep=' ')
labels = torch.tensor(labels.values, dtype=torch.float32)

# Конвертация относительных координат в абсолютные (пиксели)
width, height = img.size
labels[:, [1, 3]] *= width
labels[:, [2, 4]] *= height


for one in range(labels.shape[0]):
    z, center_x, center_y, width, height = labels[one]
    # Вычисление координат углов прямоугольника (левый верхний угол)
    left = center_x - (width / 2)
    top = center_y - (height / 2)
    # Создание прямоугольника
    rect = patches.Rectangle(
        (left, top), width, height,                         # (x, y) левого верхнего угла, ширина, высота
        linewidth=1, edgecolor='green', facecolor='none'    # толщина линии, цвет рамки, прозрачная заливка
    )
    # Добавление прямоугольника на изображение
    plt.gca().add_patch(rect)

# Отображение результата
plt.axis('off')  # Скрыть оси
plt.show()

# Использую решения Ultralytics

Вместо того, чтобы писать кастомные dataset, dataloader и функцию train

Для того, чтобы использовать предобученную YOLA, требуется подготовить файл .yaml

Сначала специальным образом подготовлю структуру папок данных:

In [ ]:
import shutil

# Пути (read only)
base_path = path_to_dataset
all_images = [f for f in os.listdir(f'{base_path}/images') if f.endswith('.jpg')]

# Разделение 80/20
train_files, val_files = train_test_split(all_images, test_size=0.2, random_state=42)

# Создаем структуру в рабочей папке
os.makedirs(DATASET_PATH / "train/images", exist_ok=True)
os.makedirs(DATASET_PATH / "train/labels", exist_ok=True)
os.makedirs(DATASET_PATH / "val/images", exist_ok=True)
os.makedirs(DATASET_PATH / "val/labels", exist_ok=True)


# Переносим файлы по нужным папкам

src_images = Path(path_to_dataset) / "images"                                           # папка куда были скачаны images
src_labels = Path(path_to_dataset) / "labels"

all_images = [f for f in os.listdir(src_images) if f.endswith(".jpg")]                  # наименования всех картинок


train_files, val_files = train_test_split(all_images, test_size=0.2, random_state=42)

for file in tqdm(train_files, desc="Копируем train данные"):
    shutil.copy(src_images / file, DATASET_PATH / f"train/images/{file}")
    name = Path(file).stem
    shutil.copy(src_labels / f"{name}.txt", DATASET_PATH / f"train/labels/{name}.txt")

for file in tqdm(val_files, desc="Копируем val данные"):
    shutil.copy(src_images / file, DATASET_PATH / f"val/images/{file}")
    name = Path(file).stem
    shutil.copy(src_labels / f"{name}.txt", DATASET_PATH / f"val/labels/{name}.txt")

вот теперь создаю файл .yaml

In [ ]:
# Создаем YAML-файл
yaml_content = f"""
path: {DATASET_PATH}
train: train/images
val: val/images
names: ['face']
nc: 1
"""

yaml_path = DATASET_PATH / "wider_face.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

print(f"Датасет подготовлен в: {DATASET_PATH}")

Подгружаю model

In [ ]:

# Load a pretrained YOLO model (recommended for training)
model = YOLO("yolo11n.pt")    # .yaml - not pretrained
model = model.to(device)

# Меняем голову модели на 1 класс
model.model.nc = 1               # Количество классов = 1 (лицо)
model.model.names = {0:'face'}     # Обновляем названия классов


In [ ]:
from ultralytics.utils.plotting import Annotator


def show_result(results):
    """
    смотрим что выводит обученная модель
    """
    # Вывод информации о детекциях
        # print(f"Найдено лиц: {len(result.boxes)}")
        # for box in result.boxes:
        #     print(f"Координаты: {box.xyxy[0].tolist()}, Уверенность: {box.conf.item():.2f}")

    plt.figure(figsize=(24, 6))
    for ind, result in enumerate(results):
        plt.subplot(1, len(results), ind+1)
        im0 = result.orig_img

        annotator = Annotator(
            im0,
            line_width=2,
            font_size=4,
            font="Arial.ttf",
            pil=False
        )
        for box in result.boxes:
            xyxy = box.xyxy[0].cpu().numpy()    # Получаю box coordinates
            conf = box.conf.item()              # Уверенность детекции
            cls = int(box.cls.item())           # Класс

            label_name = model.model.names[cls] # Название класса из model.names

            # Формирую подпись (можно добавить confidence)
            label = f"{label_name} {conf:.2f}" if conf else label_name
            annotator.box_label(
                box=xyxy,
                label=label_name,
                color=(0, 243, 68),
                txt_color=(10, 0, 10),
                rotated=False
            )

        im0_rgb = cv2.cvtColor(im0, cv2.COLOR_BGR2RGB)
        plt.imshow(im0_rgb)
        plt.axis('off')
    # plt.show()


In [ ]:
# загрузим картинку из интернета
! gdown https://images.squarespace-cdn.com/content/v1/577d1a0ce4fcb5ea7f24512e/1479230624871-W45IR9YJ7KJFG8Y1ACQD/shutterstock_292958459_GCG.jpg

In [ ]:
source1 = Path(path_to_dataset) / 'images/wider_10005.jpg'
source2 = Path(path_to_dataset) / 'images/wider_100.jpg'
source3 = Path(path_to_dataset) / 'images/wider_50.jpg'
source4 = './shutterstock_292958459_GCG.jpg'

# Run inference on the sources
results = model([source1, source2, source3, source4])

show_result(results)

можно заметить, что на данном этапе класс person модель принимает как face
поэтому дётся эту модель дообучить

In [ ]:

# Добавьте callback для отслеживания:

def on_train_epoch_start(trainer):
    print(f"Epoch {trainer.epoch + 1}/{trainer.epochs} started")
def on_train_epoch_end(trainer):
    print("Epoch finished")

# Load YOLO11n model
model = YOLO('yolo11n.pt')

# model.add_callback("on_train_epoch_start", on_train_epoch_start)
# model.add_callback("on_train_epoch_end", on_train_epoch_end)

# Train the model with single class
model.train(
    data=str(yaml_path.absolute()),
    epochs=2,
    imgsz=640,
    batch=16,                                   # Default = 16
    single_cls=True,                            # Enable single class training
    device=device,                              # Использование нескольких GPU
    plots=True,                                 # Отключение Tensorboard  # http://localhost:6006/ to monitor your training progress in real-time
    # verbose=False,                              # Главный параметр для сокращения вывода
    exist_ok=True,                              # Продолжать в существующей папке
    project=str(DATASET_PATH / "face_detection"),   # Стандартная папка для сохранения
    name='exp1',                                # Имя эксперимента
    workers=4,
    save_period=1                               # Для
)


# TensorBoard
# try:
#     %load_ext tensorboard
#     log_dir = "face_detection/exp1"
#     %tensorboard --logdir {log_dir} --host 0.0.0.0
# except Exception as e:
#     print(f"\n⚠️ TensorBoard error: {str(e)}")
#     print(f"Try running manually: tensorboard --logdir {log_dir}")

In [ ]:
# Загрузка обученной модели
model = YOLO(str(DATASET_PATH / 'face_detection/exp1/weights/best.pt'))  # путь к лучшим весам /content/face_detection/exp1/weights/best.pt
model.model.names = {0: 'face'}

# Run inference on the source
results = model([source1, source2, source3, source4])  # list of Results objects
#Смотрим
show_result(results)

# Детекция людей (Person Detection)

In [ ]:
# Загрузка обученной модели
model = YOLO('yolo11n.pt')
# Оставляю только один класс
model.model.nc = 1
model.model.names = {0: 'person'}

In [ ]:
# Define path to the image files
source1 = path_to_dataset + "/images/wider_43.jpg"
source2 = path_to_dataset + "/images/wider_154.jpg"
source3 = path_to_dataset + "/images/wider_55.jpg"

# Run inference on the source
results = model([source1, source2, source3])  # list of Results objects
#Смотрим
show_result(results)

### Детекция на видео

ни на colab, ни на kaggle данные возможности не работают в полную меру.

Детекция видео должна работать на локальной машине

In [ ]:
from IPython.display import HTML, display
from google.colab.patches import cv2_imshow  # Только для Colab

# 1. Подготовка источника (универсальный вариант)
if 'google.colab' in str(get_ipython()):
    # Для Colab используем прямой URL
    source = "https://rutube.ru/video/36fc5c318c469a404d7254f66ddfdfef/"
else:
    # Для Kaggle нужно сначала скачать видео
    !wget https://example.com/video.mp4 -O video.mp4
    source = "video.mp4"


# 2. Обработка видео с прогресс-баром
try:
    # Создаем прогресс-бар
    progress = tqdm(unit="frame")

    # Запускаем детекцию
    results = model(source, stream=True, verbose=False)  # Отключаем стандартный вывод

    # Для отображения в Colab/Kaggle
    for frame in results:
        # Обновляем прогресс-бар
        progress.update(1)

        # Получаем обработанный кадр с bounding boxes
        annotated_frame = frame.plot()  # Автоматически рисует детекции

        # Конвертируем BGR -> RGB для корректного отображения
        rgb_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)

        # Показываем кадр (для Colab)
        if 'google.colab' in str(get_ipython()):
            cv2_imshow(rgb_frame)
        else:
            # Для Kaggle/Jupyter
            display(Image.fromarray(rgb_frame))

        # Очищаем вывод между кадрами
        display.clear_output(wait=True)

except Exception as e:
    print(f"Ошибка при обработке видео: {e}")
finally:
    progress.close()

Детекция на потоковом видео с webcam

In [ ]:
cap = cv2.VideoCapture(0)  # веб-камера

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(source=cap, stream=True)  # stream=True для видео

    for result in results:
        frame = result.plot()  # рисуем bbox на кадре

    cv2.imshow('Face Detection', frame)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets
from google.colab.output import eval_js
from base64 import b64decode
import time

# Кнопка для остановки
stop_button = widgets.Button(description="Остановить")
display(stop_button)

# Флаг для остановки
stop_detection = False

def on_button_clicked(b):
    global stop_detection
    stop_detection = True
    print("Детекция остановлена")

stop_button.on_click(on_button_clicked)

# Основной цикл обработки
while not stop_detection:
    try:
        # Захватываем кадр через JS
        js = Javascript('''
            async function capture() {
                const div = document.createElement('div');
                const video = document.createElement('video');
                video.style.display = 'block';
                const stream = await navigator.mediaDevices.getUserMedia({video: true});
                video.srcObject = stream;
                await video.play();

                // Ждем стабилизации
                await new Promise(resolve => setTimeout(resolve, 200));

                const canvas = document.createElement('canvas');
                canvas.width = video.videoWidth;
                canvas.height = video.videoHeight;
                canvas.getContext('2d').drawImage(video, 0, 0);
                stream.getVideoTracks()[0].stop();
                return canvas.toDataURL('image/jpeg', 0.8);
            }
        ''')
        display(js)
        data = eval_js('capture()')

        # Декодируем изображение
        binary = b64decode(data.split(',')[1])
        img = cv2.imdecode(np.frombuffer(binary, dtype=np.uint8), -1)

        # Детекция
        results = model(img)

        # Отображение результатов
        for r in results:
            annotated_img = r.plot()  # Рисуем bbox'ы
            annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
            clear_output(wait=True)
            display(widgets.Image(value=cv2.imencode('.jpg', annotated_img)[1].tobytes()))

    except Exception as e:
        print(f"Ошибка: {e}")
        break

print("Работа завершена")

Гляну одним глазком на сегментацию:

In [ ]:
model = YOLO('yolo11n-seg.pt')                              # load a pretrained YOLO segmentation model
model.train(data='coco8-seg.yaml', epochs=5)                # train the model
results = model('https://ultralytics.com/images/bus.jpg')   # predict on an image

# Визуализация результатов
for result in results:
    # Вариант 1: Используем встроенный plot()
    plt.figure(figsize=(24, 7))
    plotted_img = result.plot()  # автоматически рисует bbox и маски
    cv2_imshow(cv2.cvtColor(plotted_img, cv2.COLOR_BGR2RGB))  # для Colab


# Кастомные функции для dataset, dataloader

кастомные будут через одну ячейку, а пока посмотрим быстрые встроенные методы, которые имеются в ultralytics, а именно

YOLODataset и build_dataloader

In [ ]:


with open(DATASET_PATH / "wider_face.yaml") as f:
    data = yaml.safe_load(f)    # data = {'path': yaml_path, 'train': 'images/train', 'val': 'images/val', 'names': ['face'], 'nc': 1}

# Создание датасетов
train_dataset = YOLODataset(
    img_path=os.path.join(data['path'], 'train'),
    data=data,
    imgsz=640,
    augment=True                # Включение аугментаций для обучения
)

val_dataset = YOLODataset(
    img_path=os.path.join(data['path'], 'val'),
    data=data,
    imgsz=640,
    augment=False               # Без аугментаций для валидации
)

# Создание DataLoader
train_dataloader = build_dataloader(train_dataset, batch=16, workers=4, shuffle=True)
val_dataloader = build_dataloader(val_dataset, batch=16, workers=4, shuffle=False)


In [ ]:
all_files = []
for path, folders, files in os.walk(images_folder_path):
    for file in tqdm(files):
        file = file.replace('.jpg', '')
        all_files.append(file)

Поделю на train/test

In [ ]:
train_files, test_files = train_test_split(all_files, test_size=0.1, shuffle=False, random_state=42)

In [ ]:
len(train_files), len(test_files)

Создаю кастомный class FaceDetectionDataset

In [ ]:
class FaceDetectionDataset(Dataset):
    def __init__(self, file_names, transforms=False):
        """
        inputs:
            files: массив с наименованиями всех фото (без .jpg)
            transforms: возможные преобразования для фото
        outputs:

        """
        self.file_names = sorted(file_names)
        self.transforms = transforms

    def __len__(self):
        return len(self.file_names)

    def image_prcss(self, file_name):
        img_path = f'{images_folder_path}/{file_name}.jpg'
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # YOLOv8 ожидает RGB
        return img


    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        file_name = self.file_names[idx]

        image = self.image_prcss(file_name)
        # Аннотации для текущего изображения
        annots = pd.read_csv(f'{labels_folder_path}/{file_name}.txt', names=['ind', 'x', 'y', 'w', 'h'], sep=' ')
        # Перевожу аннотации в numpy
        if annots.empty:
            annots = np.zeros((0, 5), dtype=np.float32)  # Если файл пустой, то будет пустой массив [0x5]
        else:
            annots = np.array(annots, dtype=np.float32)

        # Применение аугментаций
        if self.transforms:
            try:
                # Проверка и обрезка координат перед аугментациями
                annots = self._validate_annotations(annots, image.shape[:2])

                transformed = self.transforms(
                    image=image,
                    bboxes=annots[:, 1:].tolist(),
                    class_labels=annots[:, 0].tolist()
                )
                image = transformed["image"]
                annots = self._process_transformed_annotations(transformed)
            except Exception as e:
                print(f"Error transforming {file_name}: {e}")
                annots = np.zeros((0, 5), dtype=np.float32)

        # Конвертация в тензоры
        # image = torch.from_numpy(image).float().permute(2, 0, 1)
        targets = torch.from_numpy(annots).float()

        return image, targets

    def _validate_annotations(self, annots, img_shape):
        """Проверка и корректировка координат bbox"""
        if len(annots) == 0:
            return annots

        h, w = img_shape
        valid_annots = []

        for annot in annots:
            cls_id, x, y, bw, bh = annot

            # Конвертация в абсолютные координаты
            x_abs = x * w
            y_abs = y * h
            bw_abs = bw * w
            bh_abs = bh * h

            # Проверка выхода за границы!
            x_min = max(0, x_abs - bw_abs/2)
            y_min = max(0, y_abs - bh_abs/2)
            x_max = min(w, x_abs + bw_abs/2)
            y_max = min(h, y_abs + bh_abs/2)

            # Проверка валидности bbox
            if x_max <= x_min or y_max <= y_min:
                continue

            # Конвертация обратно в относительные координаты
            new_x = (x_min + x_max) / (2 * w)
            new_y = (y_min + y_max) / (2 * h)
            new_bw = (x_max - x_min) / w
            new_bh = (y_max - y_min) / h

            valid_annots.append([cls_id, new_x, new_y, new_bw, new_bh])

        return np.array(valid_annots, dtype=np.float32)

    def _process_transformed_annotations(self, transformed):
        """Обработка аннотаций после аугментаций"""
        if len(transformed["bboxes"]) == 0:
            return np.zeros((0, 5), dtype=np.float32)

        # Обрезка координат до [0, 1]
        bboxes = np.clip(transformed["bboxes"], 0.0, 1.0)
        class_labels = transformed["class_labels"]

        return np.column_stack([class_labels, bboxes])

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

"""
Применю конвейер преобразований (albumentations) для корректной подготовки данных для модели Yolo.
Конвейер из 3 операций с указанием формата boundingbox (YOLO).
Albumentations последовательно применяет все аугментации, все преобразования автоматически синхронизируют изображение и bbox
"""
transform = A.Compose([
    # A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    # A.Blur(blur_limit=3, p=0.1),
    # A.GaussNoise(var_limit=10, p=0.1),
    # A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT),
    A.Resize(640, 640),
    A.HorizontalFlip(p=0.5),
    A.Normalize(
        mean=[0, 0, 0],       # Не вычитаем среднее
        std=[1, 1, 1],        # Делим на 1 (фактически просто масштабируем [0,255] -> [0,1])
        max_pixel_value=255.0
    ),
    ToTensorV2()              # Конвертируем в тензор и меняем порядок осей (HWC -> CHW)
], bbox_params=A.BboxParams(
    format="yolo",
    label_fields=['class_labels'] # указываем Albumentations, где искать метки классов
)
                     )

Делаю Dataset

In [ ]:
train_dataset = FaceDetectionDataset(train_files, transform)
test_dataset = FaceDetectionDataset(test_files, transform)

In [ ]:
img, annot = train_dataset[0]
img.shape, annot

Делаю Dataloader's

In [ ]:
def custom_collate_fn(batch):
    """
    Обрабатывает батч из вашего датасета.
    Каждый элемент batch — это кортеж (image, targets), где:
    - image: torch.Size([3, 640, 640])
    - targets: tensor([[cls, x_center, y_center, w, h], ...])
    """
    images = []
    bboxes = []
    cls = []
    batch_idx = []

    for i, (img, targets) in enumerate(batch):
        images.append(img)

        if len(targets) > 0:
            # Разделяем классы и координаты
            cls.append(targets[:, 0])  # Классы (всегда 0 для face detection)
            boxes = targets[:, 1:]     # Координаты [x_center, y_center, w, h]

            # Конвертируем в формат [x1, y1, x2, y2]
            xyxy = torch.zeros_like(boxes)
            xyxy[:, 0] = boxes[:, 0] - boxes[:, 2] / 2  # x1 = x_center - w/2
            xyxy[:, 1] = boxes[:, 1] - boxes[:, 3] / 2  # y1 = y_center - h/2
            xyxy[:, 2] = boxes[:, 0] + boxes[:, 2] / 2  # x2 = x_center + w/2
            xyxy[:, 3] = boxes[:, 1] + boxes[:, 3] / 2  # y2 = y_center + h/2
            bboxes.append(xyxy)
            batch_idx.append(torch.full((len(targets),), i))  # Индекс изображения в батче

    # Собираем в один тензор там, где возможно
    images = torch.stack(images)  # [B, 3, 640, 640]

    return {
        'img': images,
        'bboxes': bboxes,  # list[tensor[N, 4]] (x1, y1, x2, y2)
        'cls': cls,        # list[tensor[N]] (все 0)
        'batch_idx': batch_idx  # list[tensor[N]] (индексы изображений)
    }

In [ ]:
batch_size = 16

train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=min(8, os.cpu_count()),  # Оптимальное число workers
    pin_memory=True,
    collate_fn=custom_collate_fn,
    drop_last=True                       # Игнорируем последний неполный батч
)

test_dataloader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=min(8, os.cpu_count()),  # Оптимальное число workers
    pin_memory=True,
    collate_fn=custom_collate_fn,
    drop_last=True                       # Игнорируем последний неполный батч
)

Функция для визуализации

In [ ]:
from ultralytics.utils.plotting import Annotator

def denormalize(image_tensor):
    """Денормализует изображение после A.Normalize()"""
    image = image_tensor.numpy()
    image = image.transpose(1, 2, 0)
    image = (image - image.min()) / (image.max() - image.min())
    return (image * 255).astype(np.uint8)


def plotting_image(batch):
    image = batch['img'][0]
    # Денормализация
    denormalized_img = denormalize(image)
    denormalized_img = np.ascontiguousarray(denormalized_img)

    # Получаем боксы и конвертируем в абсолютные координаты
    boxes = batch['bboxes'][0].cpu().numpy()

    h, w = denormalized_img.shape[:2]
    boxes[:, [0, 2]] *= w
    boxes[:, [1, 3]] *= h
    # Конвертируем тип координат # OpenCV требует целые числа
    boxes = boxes.astype(int)

    # Создаем аннотатор
    annotator = Annotator(denormalized_img)
    annotator.lw = 1 # толщина линии

    # Рисуем все боксы
    for box in boxes:
        # Убедимся, что координаты корректны
        if len(box) == 4:
            annotator.box_label(box=box,
                                label='face',
                                color=(0, 255, 0),
                                txt_color=(0, 0, 0))
        else:
            print(f"Пропущен некорректный бокс: {box}")

    # Визуализация
    plt.imshow(denormalized_img)
    plt.axis('off')
    # plt.show()

In [ ]:
test_batch = next(iter(train_dataloader))
# test_batch

Смотрю на один пример из train_dataloader

In [ ]:
plotting_image(test_batch)

In [ ]:
def plot_metrics(model, df):
    """Визуализация метрик из CSV-лога"""
    #'epoch', 'lr', 'val_mAP50', 'val_mAP50-95', 'val_precision', 'val_recall', 'train_box_loss'
    # Путь к файлу с метриками
    log_path = os.path.join(model.trainer.save_dir, 'results.csv')

    if os.path.exists(log_path):
        # df = pd.read_csv(log_path)

        plt.figure(figsize=(15, 5))

        # Графики метрик
        plt.subplot(1, 3, 1)
        plt.plot(df['epoch'], df['val_mAP50'], label='val mAP50')
        plt.plot(df['epoch'], df['val_mAP50-95'], label='val mAP50-95')
        plt.title('Val mAP Metrics')
        plt.legend()

        # Графики потерь
        plt.subplot(1, 3, 2)
        plt.plot(df['epoch'], df['train_box_loss'], label='train box loss')
        # plt.plot(df['epoch'], df['train/cls_loss'], label='Cls Loss')
        plt.title('Training box Losses')
        plt.legend()

        # Графики precision/recall
        plt.subplot(1, 3, 3)
        plt.plot(df['epoch'], df['val_precision'], label='val Precision')
        plt.plot(df['epoch'], df['val_recall'], label='val Recall')
        plt.title('Val Precision & Recall')
        plt.legend()

        plt.tight_layout()
        plt.show()
    else:
        print("Файл с метриками не найден!")

### Train for Face Detection - Failed